In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [6]:
train = pd.read_csv("data/raw/train.csv")
holidays = pd.read_csv("data/raw/holidays_events.csv")
stores = pd.read_csv("data/raw/stores.csv")
oil = pd.read_csv("data/raw/oil.csv")
transactions = pd.read_csv("data/raw/transactions.csv")

## Step 2

## Inspect and Understand Data

In [7]:
print("train:", train.shape)
print("stores:", stores.shape)
print("holidays:", holidays.shape)
print("oil:", oil.shape)
print("transactions:", transactions.shape)

train: (3000888, 6)
stores: (54, 5)
holidays: (350, 6)
oil: (1218, 2)
transactions: (83488, 3)


In [8]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [11]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [12]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [13]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [14]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [16]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


In [17]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


In [18]:
holidays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB


In [19]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB


In [20]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB


## check missing values

In [21]:
for name, df in {
    "train": train,
    "stores": stores,
    "holidays": holidays,
    "oil": oil,
    "transactions": transactions
}.items():
    print(f"\n{name}")
    print(df.isnull().sum())


train
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

stores
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

holidays
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

oil
date           0
dcoilwtico    43
dtype: int64

transactions
date            0
store_nbr       0
transactions    0
dtype: int64


## Missing values in `dcoilwtico` were handled using forward fill and backward fill 
## because oil prices are time-series data and nearby values are the most reasonable estimate.

In [23]:
oil["dcoilwtico"] = oil["dcoilwtico"].ffill().bfill()

In [24]:
oil.isnull().sum()

date          0
dcoilwtico    0
dtype: int64

In [22]:
train.describe(include="all")

,id,date,store_nbr,family,sales,onpromotion
count,3.000888e+06,3000888,3.000888e+06,3000888,3.000888e+06,3.000888e+06
unique,NaN,1684,NaN,33,NaN,NaN
top,NaN,2013-01-01,NaN,AUTOMOTIVE,NaN,NaN
freq,NaN,1782,NaN,90936,NaN,NaN
mean,1.500444e+06,NaN,2.750000e+01,NaN,3.577757e+02,2.602770e+00
std,8.662819e+05,NaN,1.558579e+01,NaN,1.101998e+03,1.221888e+01
min,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,0.000000e+00
25%,7.502218e+05,NaN,1.400000e+01,NaN,0.000000e+00,0.000000e+00
50%,1.500444e+06,NaN,2.750000e+01,NaN,1.100000e+01,0.000000e+00
75%,2.250665e+06,NaN,4.100000e+01,NaN,1.958473e+02,0.000000e+00


## Step 3

In [25]:
train["date"] = pd.to_datetime(train["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
oil["date"] = pd.to_datetime(oil["date"])
transactions["date"] = pd.to_datetime(transactions["date"])

In [26]:
train.info()
holidays.info()
oil.info()
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[ns]
 2   store_nbr    int64         
 3   family       object        
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 137.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[ns]
 1   type         350 non-null    object        
 2   locale       350 non-null    object        
 3   locale_name  350 non-null    object        
 4   description  350 non-null    object        
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[ns](1), object(4)

## Step 4

In [27]:
train.head(10)

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0
6,6,2013-01-01,1,CELEBRATION,0.0,0
7,7,2013-01-01,1,CLEANING,0.0,0
8,8,2013-01-01,1,DAIRY,0.0,0
9,9,2013-01-01,1,DELI,0.0,0


#### Data grain

#### Each row in `train` represents the sales of one product family in one store on one day.

#### Project grain: `store_nbr + family + date`

## Step 5 
## Basic Exploration

In [29]:
print("Min date:", train["date"].min())
print("Max date:", train["date"].max())

Min date: 2013-01-01 00:00:00
Max date: 2017-08-15 00:00:00


In [31]:
print("Number of stores:", train["store_nbr"].nunique())

Number of stores: 54


In [32]:
print("Number of families:", train["family"].nunique())

Number of families: 33


In [33]:
print("Duplicate rows:", train.duplicated().sum()) ## Duplicates values checking

Duplicate rows: 0


In [34]:
print(train.isnull().sum()) ## no missing values

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64


## Step 6 — understand the other tables

## Supporting tables

- `stores`: metadata about each store such as city, state, type, and cluster
- `holidays`: holidays and events that may affect demand
- `oil`: daily oil prices, used as an external economic signal
- `transactions`: number of daily transactions per store, useful as a demand-related feature

In [36]:
# print head of each file to understand it but we already printed above so we need to fix them at one place

In [35]:
train_clean = train.copy()
stores_clean = stores.copy()
holidays_clean = holidays.copy()
oil_clean = oil.copy()
transactions_clean = transactions.copy()

## Step 8: Ready for SQL

In [37]:
import sqlite3

In [45]:
conn = sqlite3.connect("retail.db")

In [46]:
train_clean.to_sql("sales", conn, if_exists="replace", index=False)
stores_clean.to_sql("stores", conn, if_exists="replace", index=False)
holidays_clean.to_sql("holidays", conn, if_exists="replace", index=False)
oil_clean.to_sql("oil", conn, if_exists="replace", index=False)
transactions_clean.to_sql("transactions", conn, if_exists="replace", index=False)

83488

In [48]:
# Checking my database creation
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""
pd.read_sql(query, conn)

,name
0,sales
1,stores
2,holidays
3,oil
4,transactions


## Step 9: Testing of Database 

In [49]:
query = """
SELECT *
FROM sales
LIMIT 5;
"""
pd.read_sql(query, conn)

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01 00:00:00,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01 00:00:00,1,BABY CARE,0.0,0
2,2,2013-01-01 00:00:00,1,BEAUTY,0.0,0
3,3,2013-01-01 00:00:00,1,BEVERAGES,0.0,0
4,4,2013-01-01 00:00:00,1,BOOKS,0.0,0


In [50]:
# counting rows
query = """
SELECT COUNT(*) AS total_rows
FROM sales;
"""
pd.read_sql(query, conn)

,total_rows
0,3000888


In [51]:
# count stores and families
query = """
SELECT COUNT(DISTINCT store_nbr) AS num_stores,
       COUNT(DISTINCT family) AS num_families
FROM sales;
"""
pd.read_sql(query, conn)

,num_stores,num_families
0,54,33


,date,store_nbr,total_sales,transactions
0,2013-01-01 00:00:00,25,2511.62,770
1,2013-01-02 00:00:00,1,7417.15,2111
2,2013-01-02 00:00:00,2,10266.72,2358
3,2013-01-02 00:00:00,3,24060.35,3487
4,2013-01-02 00:00:00,4,10200.08,1922
5,2013-01-02 00:00:00,5,10598.62,1903
6,2013-01-02 00:00:00,6,13520.49,2143
7,2013-01-02 00:00:00,7,11997.50,1874
8,2013-01-02 00:00:00,8,14659.33,3250
9,2013-01-02 00:00:00,9,15867.48,2940
